# Running base ORIGAMI

Runs the original ORIGAMI v2.0 C code from Python on a small test simulation, then reads the crossing-number (CN) tags it writes.

Pipeline here: **make test particles → write params file → `origamitag` → read `tag.dat`**.

In [2]:
import subprocess
import os 
import numpy as np

Folder holding the compiled ORIGAMI executables (`origamitag`, `origamihalo`, …)

In [3]:
BIN = "/Users/quinnkasdan-grollo/Documents/DESI_ORIGAMI/origami/code"

## Helpers

- **`run(prog, params)`** runs one ORIGAMI program with a parameter file and prints its output. `OMP_NUM_THREADS` sets how many cores OpenMP uses (only matters if `nsplit` > 1).
  - ORIGAMI's exit codes are unreliable (`origamitag` returns 1 on success; errors call `exit(0)`), so check the printed text instead — errors start with `Unable to ...`.
- **`read_tags(path)`** reads `tag.dat`: an int32 particle count, then one byte per particle (0 void, 1 wall, 2 filament, 3 halo).

In [4]:
def run(prog, params, threads=8):
    env = {**os.environ, "OMP_NUM_THREADS": str(threads)}
    r = subprocess.run([f"{BIN}/{prog}", params], env=env,
                       capture_output=True, text=True)
    print(r.stdout)
    if r.stderr:
        print(r.stderr)
    if r.returncode < 0:      # negative = killed by a signal, e.g. -11 is a segfault
        raise RuntimeError(f"{prog} crashed (signal {-r.returncode})")

def read_tags(path):
    with open(path, "rb") as f:
        n = np.fromfile(f, np.int32, 1)[0]
        return np.fromfile(f, np.uint8, n)


## Make a test particle file

ORIGAMI needs particles that **started on a regular grid** and then moved. This builds a fake "evolved universe" with the **Zel'dovich approximation**: put N³ particles on a grid, then shift each one by a smooth random displacement field. Where neighbouring particles overtake each other, ORIGAMI counts a crossing.

- `amp` = how evolved the field is (rms density contrast). ~1 barely folds; try 3–5 for more walls/filaments/halos.
- File format (`numfiles 0` in params): int32 count, then all x, all y, all z as float32.
- Particle order must be `index = ix + iy·N + iz·N²` — ORIGAMI infers each particle's starting grid point from its position in the list. `order="F"` does this.

In [35]:
# def make_test_particles(path, N=64, L=100.0, amp=1.0, ns=-1.5, seed=42):
#     """Grid of N^3 particles in a box of size L, displaced by a Gaussian random field.
#     amp = rms linear density contrast; bigger amp -> more shell-crossing."""
#     rng = np.random.default_rng(seed)
#
#     # wavenumbers for every grid cell, arrays indexed [ix, iy, iz]
#     k1 = 2*np.pi*np.fft.fftfreq(N, d=L/N)
#     kx, ky, kz = np.meshgrid(k1, k1, k1, indexing="ij")
#     k2 = kx**2 + ky**2 + kz**2
#     k2[0, 0, 0] = 1.0                        # avoid dividing by zero
#
#     # random density field with power spectrum P(k) ∝ k^ns
#     delta_k = np.fft.fftn(rng.standard_normal((N, N, N))) * k2**(ns/4)
#     delta_k[0, 0, 0] = 0.0                   # no mean overdensity
#     delta_k *= amp / np.real(np.fft.ifftn(delta_k)).std()
#
#     # Zel'dovich displacement: psi_k = i k delta_k / k^2
#     psi = [np.real(np.fft.ifftn(1j*k/k2*delta_k)) for k in (kx, ky, kz)]
#
#     # starting grid positions, then displace and wrap into the periodic box
#     q = np.arange(N) * (L/N)
#     qx, qy, qz = np.meshgrid(q, q, q, indexing="ij")
#     pos = [(qi + p) % L for qi, p in zip((qx, qy, qz), psi)]
#
#     # order='F' flattens with ix fastest -> index = ix + iy*N + iz*N^2
#     with open(path, "wb") as f:
#         np.array([N**3], dtype=np.int32).tofile(f)
#         for c in pos:
#             c.ravel(order="F").astype(np.float32).tofile(f)
#
# make_test_particles("test_n64.pos", N=64, L=100.0, amp=.5)

In [74]:
def make_test_walls(path, N=64, L=100.0, seed=42):
    rng = np.random.default_rng(seed)
    dx = L/N #spacing between particles 
    q1 = np.arange(N) * dx 
    
    #make N^3 cube of particles and store their x,y,z in qx, qy, qz
    qx, qy, qz = np.meshgrid(q1, q1, q1, indexing='ij') 


    #flattened starting grids, fortran order used b/c of how origami reads grid: x + yN + zN^2
    x0 = qx.ravel(order="F") 
    y0 = qy.ravel(order="F") 
    z0 = qz.ravel(order="F")

    #attract particles in 40<x<60 to x=50  with spring-like force 
    window = (qx >= 45) & (qx <= 55) 
    
    #initalizing lists to evolve 
    x = qx.copy() 
    vx = np.zeros_like(qx)

    #feel free to adjust 
    dt = 0.01
    omega = 1
    nsteps = 100000
    
    for n in range(nsteps):
        ax = -omega**2 * (x - 50) #spring force with equil. at x=50 
        ax[~window] = 0 #0 accel outside desired window 
        vx += ax*dt
        x += vx*dt 

    x = x % L #wrap x into the box 
    x1 = x.ravel(order="F") #flatten final positions (no evolution in y,z no need to flatten)

    # Printing out the expected physics
    particles_in_window = 0 
    for i in window:
        if i.all() == True:
            particles_in_window += 1 

    print(f"Particles in window: {particles_in_window}")
    print(f"Max crossing % possible: {particles_in_window/N}")
        
    with open(path, "wb") as f:
        np.array([N**3], dtype=np.int32).tofile(f)
        x1.astype(np.float32).tofile(f)
        y0.astype(np.float32).tofile(f)
        z0.astype(np.float32).tofile(f)
    
make_test_walls("../running_base_ORIGAMI/0CN_n64tag.pos")

Particles in window: 7
Max crossing % possible: 0.109375


## Run `origamitag`

Computes each particle's crossing number from `parameters_example.txt` and writes `outdir + taglabel + "tag.dat"`.


In [75]:
run("origamitag", "/Users/quinnkasdan-grollo/Documents/DESI_ORIGAMI/running_base_ORIGAMI/parameters_example.txt")

Reading parameter file /Users/quinnkasdan-grollo/Documents/DESI_ORIGAMI/running_base_ORIGAMI/parameters_example.txt
262144 particles
Calculating ORIGAMI morphology.
nhalo=0,0,0,0,0
Writing morphology file to ../runs/test/n64tag.dat.



## Read the results

`np.bincount` counts particles with tag 0, 1, 2, 3; dividing by the total gives the **fraction in each CN class** `[void, wall, filament, halo]`.

For `amp=1` this came out ≈ 97.4% void, 2.5% wall, 0.08% filament, 0.003% halo — the test universe has barely started to fold. (For comparison, Veyrat et al. 2024 find ~31/19/18/32% in AbacusSummit.)

In [76]:
tag = read_tags("../runs/test/n64tag.dat")
print(np.bincount(tag, minlength=4) / tag.size)

[1. 0. 0. 0.]
